In [1]:
import pandas as pd
import os
import glob

In [ ]:
# Structure des dossiers
DOSSIER_PARENT = r"C:\Users\user\Documents\projets\projet_plateforme\projet_plateforme\Projet_Terre-vent-feu-eau-data\data"

# Dossier source des fichiers météo géants
DOSSIER_RAW = os.path.join(DOSSIER_PARENT, "raw")

# Dossier où se trouvent les fichiers propres 
DOSSIER_PROCESSED = os.path.join(DOSSIER_PARENT, "processed")

# Chemins précis des fichiers de données
chemin_incendies_parquet = os.path.join(DOSSIER_PROCESSED, "bdiff_consolidee_phase1.parquet")
chemin_sortie_meteo_parquet = os.path.join(DOSSIER_PROCESSED, "meteo_allogee_1983_2026.parquet")

# Détection automatique des fichiers météo bruts
fichiers_meteo_trouves = glob.glob(os.path.join(DOSSIER_RAW, "QUOT_SIM2_*.csv"))

print("--- 🔍 VÉRIFICATION GÉNÉRALE DES CHEMINS ---")
print(f"1. Dossier Météo Brut (Source) : {DOSSIER_RAW}")
print(f"2. Dossier Fichiers Traités    : {DOSSIER_PROCESSED}")
print(f"3. Fichier Incendies détecté   : {os.path.basename(chemin_incendies_parquet)} (Doit être dans 'processed')")
print(f"4. Futur Fichier Météo créé    : {os.path.basename(chemin_sortie_meteo_parquet)}")

print("\n--- 📄 FILES MÉTÉO À TRAITER ---")
if len(fichiers_meteo_trouves) == 0:
    print("⚠️ Attention : Aucun fichier commencé par 'QUOT_SIM2_' n'a été trouvé dans le dossier 'raw'.")
else:
    for f in fichiers_meteo_trouves:
        print(f" - {os.path.basename(f)}")

--- 🔍 VÉRIFICATION GÉNÉRALE DES CHEMINS ---
1. Dossier Météo Brut (Source) : C:\Users\user\Documents\projets\projet_plateforme\projet_plateforme\Projet_Terre-vent-feu-eau-data\data\raw
2. Dossier Fichiers Traités    : C:\Users\user\Documents\projets\projet_plateforme\projet_plateforme\Projet_Terre-vent-feu-eau-data\data\processed
3. Fichier Incendies détecté   : bdiff_consolidee_phase1.parquet (Doit être dans 'processed')
4. Futur Fichier Météo créé    : meteo_allogee_1983_2026.parquet

--- 📄 FILES MÉTÉO À TRAITER ---
 - QUOT_SIM2_1980-1989.csv
 - QUOT_SIM2_1990-1999.csv
 - QUOT_SIM2_2000-2009.csv
 - QUOT_SIM2_2010-2019.csv
 - QUOT_SIM2_2020-2026.csv


## Filtrage et l'Allègement de la Météo

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
import os

colonnes_utiles = ['LAMBX', 'LAMBY', 'DATE', 'PRELIQ', 'PRENEI', 'T', 'FF', 'HU'] 

print("🚀 Lancement du traitement en flux continu (Streaming vers Parquet)...")
print("RAM surveillée : Écriture directe sur le disque dur.\n")

writer = None

for fichier_chemin in fichiers_meteo_trouves:
    nom_fichier = os.path.basename(fichier_chemin)
    print(f"📖 Flux en cours : {nom_fichier} ...")
    
    blocs = pd.read_csv(
        fichier_chemin, 
        sep=';', 
        usecols=colonnes_utiles, 
        chunksize=200000,  
        low_memory=False
    )
    
    for chunk in blocs:
        # Conversion rapide de la date
        chunk['DATE'] = pd.to_datetime(chunk['DATE'], format='%Y%m%d', errors='coerce')
        
        # Application du filtre 1983
        chunk_filtre = chunk[chunk['DATE'] >= '1983-01-01']
        
        if not chunk_filtre.empty:
            # Conversion du bloc Pandas en Table PyArrow (très léger)
            table = pa.Table.from_pandas(chunk_filtre, preserve_index=False)
            
            # Initialisation du fichier Parquet au tout premier bloc rencontré
            if writer is None:
                writer = pq.ParquetWriter(chemin_sortie_meteo_parquet, table.schema, compression='snappy')
            
            # Écriture immédiate du bloc sur le disque dur 
            writer.write_table(table)

# Fermeture propre du fichier final
if writer is not None:
    writer.close()

print(f"\n✅ Traitement terminé avec succès !")
print(f"Le fichier unique compressé est disponible ici : {chemin_sortie_meteo_parquet}")

🚀 Lancement du traitement en flux continu (Streaming vers Parquet)...
RAM surveillée : Écriture directe sur le disque dur.

📖 Flux en cours : QUOT_SIM2_1980-1989.csv ...
📖 Flux en cours : QUOT_SIM2_1990-1999.csv ...
📖 Flux en cours : QUOT_SIM2_2000-2009.csv ...
📖 Flux en cours : QUOT_SIM2_2010-2019.csv ...
📖 Flux en cours : QUOT_SIM2_2020-2026.csv ...

✅ Victoire ! Traitement terminé avec succès !
Le fichier unique compressé est disponible ici : C:\Users\user\Documents\projets\projet_plateforme\projet_plateforme\Projet_Terre-vent-feu-eau-data\data\processed\meteo_allogee_1983_2026.parquet


## Inspection et Validation Statistiques

In [ ]:
print("--- 📊 INSPECTION OPTIMISÉE DU FICHIER MÉTÉO ---")

# Ouverture du fichier sans aucun chargement en RAM
fichier_parquet = pq.ParquetFile(chemin_sortie_meteo_parquet)
nb_lignes = fichier_parquet.metadata.num_rows
nb_colonnes = fichier_parquet.metadata.num_columns

print(f"Nombre total de lignes : {nb_lignes:,}".replace(",", " "))
print(f"Nombre de variables (colonnes) : {nb_colonnes}")

# Lecture du 1er groupe de lignes (Row Group 0)
# et on extrait les 5 premières lignes en mémoire
premier_bloc = fichier_parquet.read_row_group(0)
df_apercu = premier_bloc.slice(0, 5).to_pandas()

print("\n--- 🔍 APERÇU DES 5 PREMIÈRES LIGNES ---")
display(df_apercu)

print("\n--- 💾 TYPES DES DONNÉES ---")
print(df_apercu.dtypes)

# Plage temporelle
date_min = df_apercu['DATE'].min()
print(f"\n📅 Date de début aperçue : {date_min.strftime('%Y-%m-%d')}")

--- 📊 INSPECTION OPTIMISÉE DU FICHIER MÉTÉO ---
Nombre total de lignes : 157 154 204
Nombre de variables (colonnes) : 8

--- 🔍 APERÇU DES 5 PREMIÈRES LIGNES ---


,LAMBX,LAMBY,DATE,PRENEI,PRELIQ,T,FF,HU
0,600,24010,1983-01-01,0.0,3.2,10.1,4.7,96.1
1,600,24010,1983-01-02,0.0,1.0,10.0,2.9,94.1
2,600,24010,1983-01-03,0.0,2.3,13.2,8.9,95.2
3,600,24010,1983-01-04,0.0,3.1,12.7,7.3,92.3
4,600,24010,1983-01-05,0.0,0.4,14.7,9.3,95.5



--- 💾 TYPES DES DONNÉES ---
LAMBX              int64
LAMBY              int64
DATE      datetime64[us]
PRENEI           float64
PRELIQ           float64
T                float64
FF               float64
HU               float64
dtype: object

📅 Date de début aperçue : 1983-01-01


In [ ]:
import numpy as np

print("🤖 Initialisation du générateur de Negative Sampling Industriel...")

# =========================================================================
# CHARGEMENT SÉCURISÉ DU RÉFÉRENTIEL DES COMMUNES
# =========================================================================
print("🔍 Recherche du référentiel des communes...")
recherche_chemin = os.path.join(DOSSIER_PARENT, "**", "communes-france-2026.csv")
fichiers_trouves = glob.glob(recherche_chemin, recursive=True)

if not fichiers_trouves:
    raise FileNotFoundError("❌ Le fichier 'communes-france-2026.csv' est introuvable.")

chemin_referentiel = fichiers_trouves[0]
df_communes_local = pd.read_csv(chemin_referentiel, usecols=['code_insee'], dtype={'code_insee': str})
df_communes_local['code_insee'] = df_communes_local['code_insee'].str.zfill(5)

communes_uniques = df_communes_local['code_insee'].unique()
print(f"✅ Référentiel chargé : {len(communes_uniques):,} communes uniques identifiées.".replace(",", " "))

# =========================================================================
# CHARGEMENT DE L'HISTORIQUE DES INCENDIES (BDIFF)
# =========================================================================
print("📖 Chargement de la base historique BDIFF pour le marquage...")
df_incendies_local = pd.read_parquet(chemin_incendies_parquet, engine='pyarrow')

# Normalisation de la colonne de date et du code INSEE
df_incendies_local['DATE_NORMALISEE'] = pd.to_datetime(df_incendies_local['Date de première alerte'], errors='coerce').dt.normalize()
df_incendies_local['code_insee'] = df_incendies_local['Code INSEE'].astype(str).str.zfill(5)

# Extraction des couples (Commune, Date) où un incendie a VRAIMENT eu lieu
faits_incendies = set(zip(df_incendies_local['code_insee'], df_incendies_local['DATE_NORMALISEE']))
print(f"🔥 {len(faits_incendies):,} événements 'Feu' enregistrés historiquement.".replace(",", " "))

# =========================================================================
# GÉNÉRATION DE LA GRILLE CIBLE (NEGATIVE SAMPLING)
# =========================================================================
# Définition de la plage temporelle (Fenêtre industrielle d'apprentissage)
dates_apprentissage = pd.date_range(start="2020-05-01", end="2025-09-30", freq="D")
print(f"📅 Génération sur la plage : {dates_apprentissage.min().strftime('%Y-%m-%d')} au {dates_apprentissage.max().strftime('%Y-%m-%d')}")

print("📐 Construction de la matrice cartésienne (Communes × Jours)...")
index_multi = pd.MultiIndex.from_product(
    [communes_uniques, dates_apprentissage], 
    names=['code_insee', 'DATE']
)
df_grille_cible = pd.DataFrame(index=index_multi).reset_index()

# =========================================================================
# MARQUAGE DE LA TARGET (0 = Pas de feu, 1 = Feu)
# =========================================================================
print("🎯 Marquage des cibles (Target Labeling)...")
df_grille_cible['TARGET'] = np.where(
    df_grille_cible.set_index(['code_insee', 'DATE']).index.isin(faits_incendies), 
    1, 
    0
)

print("\n--- 📊 BILAN DU NEGATIVE SAMPLING ---")
print(f"Dimensions de la matrice : {df_grille_cible.shape[0]:,}".replace(",", " ") + " lignes.")
print(df_grille_cible['TARGET'].value_counts(normalize=False))
print(df_grille_cible['TARGET'].value_counts(normalize=True) * 100)

🤖 Initialisation du générateur de Negative Sampling Industriel...
🔍 Recherche du référentiel des communes...
✅ Référentiel chargé : 34 868 communes uniques identifiées.
📖 Chargement de la base historique BDIFF pour le marquage...
🔥 122 623 événements 'Feu' enregistrés historiquement.
📅 Génération sur la plage : 2020-05-01 au 2025-09-30
📐 Construction de la matrice cartésienne (Communes × Jours)...
🎯 Marquage des cibles (Target Labeling)...

--- 📊 BILAN DU NEGATIVE SAMPLING ---
Dimensions de la matrice : 69 003 772 lignes.
TARGET
0    68973191
1       30581
Name: count, dtype: int64
TARGET
0    99.955682
1     0.044318
Name: proportion, dtype: float64


**Sauvegarder la matrice de Negative Sampling**

In [10]:
chemin_sortie_grille = os.path.join(DOSSIER_PROCESSED, "target_matrix_negative_sampling.parquet")
print(f"💾 Sauvegarde de la matrice cible ({df_grille_cible.shape[0]:,} lignes)...")

df_grille_cible.to_parquet(chemin_sortie_grille, engine='pyarrow', index=False)
print(f"✅ Matrice enregistrée avec succès : {chemin_sortie_grille}")

💾 Sauvegarde de la matrice cible (69,003,772 lignes)...
✅ Matrice enregistrée avec succès : C:\Users\user\Documents\projets\projet_plateforme\projet_plateforme\Projet_Terre-vent-feu-eau-data\data\processed\target_matrix_negative_sampling.parquet
